# 🦿 G1 ヒューマノイドロボットの歩行を学習させよう！

このノートブックでは Unitree **G1** — 29 関節を持つ二足歩行ロボット — に **自力で歩く** ことを学習させます。

## Go2（四足）との違い

| 項目 | Go2 (四足) | **G1 (二足)** |
|------|-----------|---------------|
| 関節数 | 12 | **29** |
| 脚の本数 | 4 | **2** |
| 重心高さ | ~0.3 m | **~0.8 m** |
| 転倒リスク | 低い | **高い** |
| 観測次元 | 45 | **98** |

二足歩行は四足歩行より格段に難しい！ でも報酬を上手に設計すれば、AI は自然な歩き方を習得できます。

## 参考文献
- [unitree_rl_mjlab](https://github.com/unitreerobotics/unitree_rl_mjlab) — Unitree 公式の MuJoCo RL フレームワーク
- [MuJoCo Menagerie G1](https://github.com/google-deepmind/mujoco_menagerie/tree/main/unitree_g1) — G1 MuJoCo モデル

## 流れ
1. 🔧 セットアップ（G1 モデルを取得）
2. 📚 G1 の関節構造を理解する
3. 🎛️ 報酬パラメータを設定する
4. 🚀 歩行を学習させる
5. 🎬 結果を動画で確認する
6. 💾 モデルを保存する

In [ ]:
#@title 🔧 セットアップ（最初に一度だけ実行してください）

import subprocess, sys, os

# ── 1. レンダリングバックエンド設定 ────────────────────────────────────────
_has_gpu = subprocess.run('nvidia-smi', shell=True, capture_output=True).returncode == 0
if _has_gpu:
    _ICD = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
    os.makedirs(os.path.dirname(_ICD), exist_ok=True)
    if not os.path.exists(_ICD):
        with open(_ICD, 'w') as _f:
            _f.write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}\n')
    os.environ['MUJOCO_GL'] = 'egl'
    print('🖥️  GPU 検出: EGL レンダリング')
else:
    subprocess.run('apt-get install -y -q libosmesa6', shell=True, check=False)
    os.environ['MUJOCO_GL'] = 'osmesa'
    print('💻  CPU モード: OSMesa レンダリング')

# ffmpeg
subprocess.run(
    'command -v ffmpeg >/dev/null || (apt-get update -qq && apt-get install -y -q ffmpeg)',
    shell=True, check=False)

# ── 2. ライブラリインストール ───────────────────────────────────────────────
print('\nライブラリをインストール中...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'mujoco', 'gymnasium', 'stable-baselines3[extra]',
    'mediapy', 'tqdm', 'matplotlib', 'pandas'], check=True)

# ── 3. RoboQuest2026 リポジトリ ────────────────────────────────────────────
if not os.path.exists('/content/RoboQuest2026/.git'):
    print('\nRoboQuest2026 をダウンロード中...')
    subprocess.run(['git', 'clone', '-q',
        'https://github.com/SingularityBattleQuest/RoboQuest2026.git',
        '/content/RoboQuest2026'], check=True)
else:
    subprocess.run(['git', '-C', '/content/RoboQuest2026', 'pull', 'origin', 'main', '-q'], check=False)

os.chdir('/content/RoboQuest2026')
if '/content/RoboQuest2026' not in sys.path:
    sys.path.insert(0, '/content/RoboQuest2026')

# ── 4. G1 MuJoCo モデルを取得 (MuJoCo Menagerie) ─────────────────────────
print('\nG1 モデルを取得中 (MuJoCo Menagerie)...')
if not os.path.exists('/content/mujoco_menagerie/.git'):
    subprocess.run([
        'git', 'clone', '--depth=1', '--filter=blob:none', '--sparse',
        'https://github.com/google-deepmind/mujoco_menagerie.git',
        '/content/mujoco_menagerie'
    ], check=True)
    subprocess.run([
        'git', '-C', '/content/mujoco_menagerie',
        'sparse-checkout', 'set', 'unitree_g1'
    ], check=True)
else:
    print('  スキップ (既存)')

G1_SCENE_XML = '/content/mujoco_menagerie/unitree_g1/scene.xml'
assert os.path.exists(G1_SCENE_XML), f'G1 モデルが見つかりません: {G1_SCENE_XML}'
print('\n✅ セットアップ完了！次のセルへ進んでください。')

In [ ]:
#@title 📁 チーム名と Google Drive 接続

#@markdown チーム名を入力してください（モデルの保存先になります）
team_name = '自分のチーム名' #@param {type:"string"}

from google.colab import drive
print('Google Drive に接続します...')
drive.mount('/content/drive')

import os
SAVE_DIR = f'/content/drive/MyDrive/RoboQuest2026/{team_name}/g1'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'\n✅ 保存先: {SAVE_DIR}')

## 📚 G1 ロボットを知ろう

### 関節構造（29 DOF）

```
G1 関節マップ
─────────────────────────────────────────
【下半身】左脚 6 関節 + 右脚 6 関節 = 12
  hip_pitch  ← 前後の振り脚（歩行の主役）
  hip_roll   ← 左右の安定
  hip_yaw    ← 水平回転
  knee       ← 膝の曲げ伸ばし
  ankle_pitch← つま先の上下
  ankle_roll ← 足首の内外傾き

【腰】3 関節
  waist_yaw / waist_roll / waist_pitch

【上半身】左腕 7 関節 + 右腕 7 関節 = 14
  shoulder_pitch/roll/yaw, elbow
  wrist_roll/pitch/yaw
─────────────────────────────────────────
合計: 12 + 3 + 14 = 29 DOF
```

### 観測空間（98 次元）

| インデックス | 内容 | 次元数 |
|-------------|------|-------|
| [0:3]       | 速度コマンド (vx, vy, ω) | 3 |
| [3:6]       | 胴体の角速度 xyz | 3 |
| [6:9]       | 重力ベクトル（ボディフレーム） | 3 |
| [9:11]      | **歩行位相 (sin, cos)** ← 交互歩行の鍵！ | 2 |
| [11:40]     | 関節角度（立位からの差） | 29 |
| [40:69]     | 関節の角速度 | 29 |
| [69:98]     | 前回の行動 | 29 |

### 💡 歩行位相って何？

二足歩行は「左足・右足を交互に出す」リズムが不可欠です。  
位相信号 `[sin(2πt/T), cos(2πt/T)]` を観測に加えることで、
AI がどのタイミングでどちらの足を上げるべきかを学びやすくなります。  
（参考: [unitree_rl_mjlab `phase()` 関数](https://github.com/unitreerobotics/unitree_rl_mjlab)）

In [ ]:
# ============================================================
#  G1 歩行環境 — G1WalkEnv
#
#  設計方針:
#   - unitree_rl_mjlab の velocity task を Gymnasium ラッパーに移植
#   - MuJoCo Menagerie の G1 position actuator モデルを使用
#   - 歩行位相 (phase) を観測に加えて交互歩行を促進
#   - 報酬: 速度追跡 + 姿勢安定 + 交互歩行リズム + スムーズネス
# ============================================================

import math
import mujoco
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from dataclasses import dataclass
from typing import Optional


# ── G1 定数 ──────────────────────────────────────────────────────────────────

# MuJoCo Menagerie「stand」キーフレームの制御目標値
# 関節順: 左脚(6) + 右脚(6) + 腰(3) + 左腕(7) + 右腕(7) = 29 DOF
G1_STANDING_CTRL = np.array([
    # 左脚: hip_pitch, hip_roll, hip_yaw, knee, ankle_pitch, ankle_roll
    0.0,  0.0, 0.0, 0.0, 0.0, 0.0,
    # 右脚: hip_pitch, hip_roll, hip_yaw, knee, ankle_pitch, ankle_roll
    0.0,  0.0, 0.0, 0.0, 0.0, 0.0,
    # 腰: yaw, roll, pitch
    0.0,  0.0, 0.0,
    # 左腕: shoulder_pitch, shoulder_roll, shoulder_yaw, elbow, wrist×3
    0.2,  0.2, 0.0, 1.28, 0.0, 0.0, 0.0,
    # 右腕: shoulder_pitch, shoulder_roll, shoulder_yaw, elbow, wrist×3
    0.2, -0.2, 0.0, 1.28, 0.0, 0.0, 0.0,
], dtype=np.float64)

# 関節グループ別行動スケール
# unitree_rl_mjlab: G1_ACTION_SCALE = 0.25 * effort_limit / stiffness
G1_ACTION_SCALE = np.array([
    # 左脚 (大きめ — 歩行の主役)
    0.40, 0.40, 0.30, 0.40, 0.25, 0.20,
    # 右脚
    0.40, 0.40, 0.30, 0.40, 0.25, 0.20,
    # 腰 (小さめ — 安定性のため)
    0.20, 0.15, 0.15,
    # 左腕 (アームスイング)
    0.30, 0.25, 0.25, 0.25, 0.15, 0.10, 0.10,
    # 右腕
    0.30, 0.25, 0.25, 0.25, 0.15, 0.10, 0.10,
], dtype=np.float64)

# 足（ankle）ボディ名 — 接地判定に使用
FOOT_BODY_NAMES = ['left_ankle_roll_link', 'right_ankle_roll_link']

G1_N_JOINTS       = 29     # 関節（アクチュエータ）数
G1_STANDING_H     = 0.79   # 立位重心高さ [m]
G1_FALL_H         = 0.50   # 転倒閾値 [m]
G1_CTRL_DT        = 0.02   # 制御周期 [s] (50 Hz)
G1_GAIT_PERIOD    = 0.6    # 歩行周期 [s] (unitree_rl_mjlab 参照)
G1_SUBSTEPS       = 5      # 物理サブステップ数


@dataclass
class G1RewardConfig:
    """G1 歩行の報酬重み設定。

    unitree_rl_mjlab の G1 velocity task を参考に設計:
    - 速度追跡: メイン報酬（Gaussian 型）
    - 姿勢・高さ: 安定性ペナルティ
    - 歩行リズム: 交互歩行ボーナス
    - スムーズネス: アクション変化・トルクペナルティ
    """
    # ── 速度追跡（主目標） ───────────────────────────────────────────────────
    lin_vel_weight: float = 1.5     # 線速度追跡の重み
    ang_vel_weight: float = 0.75    # 角速度追跡の重み

    # ── 安定性ペナルティ ─────────────────────────────────────────────────────
    orientation_weight: float = -2.0  # 傾きペナルティ（重力ベクトルの xy 成分）
    height_weight: float = -2.0       # 高さ偏差ペナルティ

    # ── 歩行リズム ───────────────────────────────────────────────────────────
    gait_weight: float = 0.5          # 交互歩行ボーナス

    # ── エネルギー効率 ───────────────────────────────────────────────────────
    action_rate_weight: float = -0.05  # 行動変化ペナルティ（スムーズな動き）
    torques_weight: float = -1e-4      # トルクペナルティ（省エネ）

    # ── 終了ペナルティ ───────────────────────────────────────────────────────
    fall_penalty: float = 20.0


class G1WalkEnv(gym.Env):
    """G1 ヒューマノイドロボットの二足歩行環境。

    観測 (98 次元):
      [0:3]   速度コマンド (vx, vy, omega)
      [3:6]   胴体角速度 xyz
      [6:9]   重力方向ベクトル（ボディフレーム）
      [9:11]  歩行位相 (sin, cos)
      [11:40] 関節角度（立位からの相対値）
      [40:69] 関節角速度
      [69:98] 前回行動

    行動 (29 次元):
      各関節の目標オフセット [-1, 1] → G1_ACTION_SCALE でスケール
    """

    metadata = {'render_modes': ['rgb_array'], 'render_fps': 50}

    def __init__(
        self,
        reward_config: Optional[G1RewardConfig] = None,
        max_episode_steps: int = 1000,
        render_mode: Optional[str] = None,
        xml_path: Optional[str] = None,
        randomize_cmd: bool = True,
    ):
        super().__init__()
        self.reward_config = reward_config or G1RewardConfig()
        self.max_episode_steps = max_episode_steps
        self.render_mode = render_mode
        self.randomize_cmd = randomize_cmd

        xml = xml_path or G1_SCENE_XML
        self.model = mujoco.MjModel.from_xml_path(xml)
        self.data = mujoco.MjData(self.model)

        # 「stand」キーフレーム ID
        self._key_id = mujoco.mj_name2id(
            self.model, mujoco.mjtObj.mjOBJ_KEY, 'stand')

        # 足ボディ ID
        self._foot_body_ids = [
            mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_BODY, name)
            for name in FOOT_BODY_NAMES
            if mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_BODY, name) >= 0
        ]

        # 観測: 3+3+3+2 + 29*3 = 98 次元
        obs_dim = 3 + 3 + 3 + 2 + G1_N_JOINTS * 3
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32)
        self.action_space = spaces.Box(
            low=-1.0, high=1.0, shape=(G1_N_JOINTS,), dtype=np.float32)

        self._last_action = np.zeros(G1_N_JOINTS)
        self._vel_cmd = np.zeros(3)
        self._step_count = 0

        if render_mode == 'rgb_array':
            self._renderer = mujoco.Renderer(self.model, height=480, width=640)
        else:
            self._renderer = None

    # ── 公開 API ──────────────────────────────────────────────────────────────

    def set_vel_cmd(self, vx: float, vy: float, omega: float) -> None:
        """外部から速度コマンドを設定する（ビューアー等との連携用）。"""
        self._vel_cmd = np.array([vx, vy, omega])

    # ── 環境 API ──────────────────────────────────────────────────────────────

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        mujoco.mj_resetData(self.model, self.data)

        # 「stand」キーフレームから初期化
        if self._key_id >= 0:
            mujoco.mj_resetDataKeyframe(self.model, self.data, self._key_id)

        # 微小ノイズで初期状態を多様化
        self.data.qpos[7:] += self.np_random.uniform(-0.05, 0.05, G1_N_JOINTS)
        self.data.qvel[:] = 0.0
        mujoco.mj_forward(self.model, self.data)

        # 速度コマンドをランダムサンプリング
        if self.randomize_cmd:
            self._vel_cmd = np.array([
                self.np_random.uniform(-1.0, 1.0),   # vx [m/s]
                self.np_random.uniform(-0.5, 0.5),   # vy [m/s]
                self.np_random.uniform(-0.8, 0.8),   # omega [rad/s]
            ])

        self._step_count = 0
        self._last_action = np.zeros(G1_N_JOINTS)
        return self._get_obs().astype(np.float32), {}

    def step(self, action: np.ndarray):
        action = np.clip(action, -1.0, 1.0)

        # 目標関節角度 = 立位姿勢 + 行動オフセット
        q_target = G1_STANDING_CTRL + action * G1_ACTION_SCALE
        q_target = np.clip(
            q_target,
            self.model.actuator_ctrlrange[:, 0],
            self.model.actuator_ctrlrange[:, 1],
        )
        self.data.ctrl[:] = q_target

        # 物理ステップ (サブステップ × 5 = 制御周期 20ms)
        for _ in range(G1_SUBSTEPS):
            mujoco.mj_step(self.model, self.data)

        self._step_count += 1
        obs = self._get_obs().astype(np.float32)
        reward = self._compute_reward(action)
        terminated = self._is_terminated()
        truncated = self._step_count >= self.max_episode_steps

        if terminated:
            reward -= self.reward_config.fall_penalty

        self._last_action = action.copy()
        return obs, reward, terminated, truncated, {}

    def render(self):
        if self._renderer is None:
            return None
        self._renderer.update_scene(self.data)
        return self._renderer.render()

    def close(self):
        if self._renderer is not None:
            self._renderer.close()

    # ── 観測 ──────────────────────────────────────────────────────────────────

    def _get_obs(self) -> np.ndarray:
        # 胴体角速度
        ang_vel = self.data.qvel[3:6].copy()
        # 重力ベクトル（ボディフレーム）
        proj_grav = self._projected_gravity()
        # 歩行位相 — 交互歩行のタイミングを AI に伝える
        t = self._step_count * G1_CTRL_DT
        phi = (t % G1_GAIT_PERIOD) / G1_GAIT_PERIOD
        phase = np.array([math.sin(2 * math.pi * phi),
                          math.cos(2 * math.pi * phi)])
        # 関節状態（立位からの相対値）
        jpos = self.data.qpos[7:7 + G1_N_JOINTS] - G1_STANDING_CTRL
        jvel = self.data.qvel[6:6 + G1_N_JOINTS].copy()

        return np.concatenate([
            self._vel_cmd, ang_vel, proj_grav, phase,
            jpos, jvel, self._last_action,
        ])

    def _projected_gravity(self) -> np.ndarray:
        """重力ベクトル [0,0,-1] をボディフレームへ逆回転。"""
        qw, qx, qy, qz = self.data.qpos[3:7]
        g_w = np.array([0.0, 0.0, -1.0])
        R = np.array([
            [1-2*(qy**2+qz**2),  2*(qx*qy+qw*qz),  2*(qx*qz-qw*qy)],
            [2*(qx*qy-qw*qz),  1-2*(qx**2+qz**2),   2*(qy*qz+qw*qx)],
            [2*(qx*qz+qw*qy),    2*(qy*qz-qw*qx),  1-2*(qx**2+qy**2)],
        ])
        return R.T @ g_w

    # ── 報酬 ──────────────────────────────────────────────────────────────────

    def _compute_reward(self, action: np.ndarray) -> float:
        cfg = self.reward_config

        # 1. 線速度追跡（Gaussian: exp(-error / std²)）
        lin_vel_xy = self.data.qvel[:2]
        lin_err = float(np.sum((self._vel_cmd[:2] - lin_vel_xy) ** 2))
        r_lin = cfg.lin_vel_weight * math.exp(-lin_err / 0.25)

        # 2. 角速度追跡
        ang_vel_z = float(self.data.qvel[5])
        ang_err = (self._vel_cmd[2] - ang_vel_z) ** 2
        r_ang = cfg.ang_vel_weight * math.exp(-ang_err / 0.5)

        # 3. 姿勢安定（重力ベクトルの xy 傾き）
        proj_grav = self._projected_gravity()
        r_orient = cfg.orientation_weight * float(np.sum(proj_grav[:2] ** 2))

        # 4. 高さ維持（立位高さからの逸脱）
        height_err = (float(self.data.qpos[2]) - G1_STANDING_H) ** 2
        r_height = cfg.height_weight * height_err

        # 5. 交互歩行リズム（位相に合わせた左右の足接地）
        r_gait = self._gait_rhythm_reward(cfg)

        # 6. 行動変化ペナルティ（急な動き防止）
        r_rate = cfg.action_rate_weight * float(np.sum((action - self._last_action) ** 2))

        # 7. トルクペナルティ（省エネ）
        r_torque = cfg.torques_weight * float(np.sum(self.data.actuator_force ** 2))

        return r_lin + r_ang + r_orient + r_height + r_gait + r_rate + r_torque

    def _gait_rhythm_reward(self, cfg: G1RewardConfig) -> float:
        """交互歩行リズム報酬。

        位相 0~0.5 では左足接地・右足浮上を期待し、
        位相 0.5~1.0 では右足接地・左足浮上を期待する。
        unitree_rl_mjlab の foot_air_time 報酬を簡略化して実装。
        """
        if cfg.gait_weight == 0 or len(self._foot_body_ids) < 2:
            return 0.0

        t = self._step_count * G1_CTRL_DT
        phi = (t % G1_GAIT_PERIOD) / G1_GAIT_PERIOD

        # 各足の高さ（地面から ankle roll リンクまでの距離）
        lh = float(self.data.xpos[self._foot_body_ids[0]][2])  # 左足
        rh = float(self.data.xpos[self._foot_body_ids[1]][2])  # 右足

        ground_th = 0.06  # 接地とみなす高さ閾値 [m]
        left_contact  = lh < ground_th
        right_contact = rh < ground_th

        # 期待接地パターン: 位相前半→左接地、後半→右接地
        want_left  = phi < 0.5
        want_right = phi >= 0.5

        score = 0.0
        score += 1.0 if (left_contact == want_left) else 0.0
        score += 1.0 if (right_contact == want_right) else 0.0
        return cfg.gait_weight * score * 0.5

    # ── 終了判定 ──────────────────────────────────────────────────────────────

    def _is_terminated(self) -> bool:
        return bool(self.data.qpos[2] < G1_FALL_H)

print('✅ G1WalkEnv クラスを定義しました')

In [ ]:
# 環境を作成して基本情報を確認しよう
import mujoco

env = G1WalkEnv()
obs, _ = env.reset()

print('=' * 50)
print('G1WalkEnv 基本情報')
print('=' * 50)
print(f'観測空間: {env.observation_space.shape}  ({env.observation_space.shape[0]} 次元)')
print(f'行動空間: {env.action_space.shape}  ({env.action_space.shape[0]} 関節)')
print(f'初期高さ: {env.data.qpos[2]:.3f} m')
print(f'足ボディ数: {len(env._foot_body_ids)}')
print()

# 関節名を表示
print('アクチュエータ（制御する関節）一覧:')
groups = {
    '左脚': list(range(0, 6)),
    '右脚': list(range(6, 12)),
    '腰  ': list(range(12, 15)),
    '左腕': list(range(15, 22)),
    '右腕': list(range(22, 29)),
}
for grp_name, ids in groups.items():
    names = [
        mujoco.mj_id2name(env.model, mujoco.mjtObj.mjOBJ_ACTUATOR, i)
        for i in ids
    ]
    print(f'  [{grp_name}] {names}')

print()
print('観測値の内訳 (最初の値):')
labels = [
    ('速度コマンド (vx,vy,ω)', obs[0:3]),
    ('胴体角速度', obs[3:6]),
    ('重力ベクトル', obs[6:9]),
    ('歩行位相 (sin,cos)', obs[9:11]),
    ('関節角度 [0:5]', obs[11:16]),
]
for label, vals in labels:
    print(f'  {label}: {np.round(vals, 3)}')

env.close()
print('\n✅ 環境確認完了！')

## 🎛️ フェーズ1: 報酬パラメータを設定しよう

G1 の報酬は大きく **3 グループ** に分かれます。

### グループ A: 速度追跡（主目標）
| パラメータ | 説明 | 推奨値 |
|------------|------|-------|
| `lin_vel_weight` | 前後・横への速度追跡の強さ | 1.0〜2.0 |
| `ang_vel_weight` | 回転速度追跡の強さ | 0.5〜1.0 |

### グループ B: 安定性（転ばないように！）
| パラメータ | 説明 | 推奨値 |
|------------|------|-------|
| `orientation_weight` | 傾きペナルティ | -1.0〜-3.0 |
| `height_weight` | 高さ維持ペナルティ | -1.0〜-3.0 |

### グループ C: 歩行品質
| パラメータ | 説明 | 推奨値 |
|------------|------|-------|
| `gait_weight` | 交互歩行ボーナス（左右のリズム） | 0.3〜1.0 |
| `action_rate_weight` | 行動の急変防止 | -0.01〜-0.1 |

---
💡 **ヒント:** まずデフォルト値で試してみましょう！  
安定性を高めたい → `orientation_weight` を大きな負にする  
速く歩かせたい → `lin_vel_weight` を大きくする  
リズミカルに歩かせたい → `gait_weight` を上げる

In [ ]:
#@title 🎛️ G1 報酬パラメータの設定

#@markdown ### グループ A: 速度追跡
#@markdown **前後・横速度をどれだけ正確に追うか**（大きいほど速さを重視）
lin_vel_weight = 1.5 #@param {type:"slider", min:0.5, max:3.0, step:0.1}

#@markdown **回転速度をどれだけ正確に追うか**
ang_vel_weight = 0.75 #@param {type:"slider", min:0.0, max:2.0, step:0.05}

#@markdown ---
#@markdown ### グループ B: 安定性（マイナス = ペナルティ）
#@markdown **傾きペナルティ**（大きな負 → 倒れないことを優先）
orientation_weight = -2.0 #@param {type:"slider", min:-5.0, max:0.0, step:0.1}

#@markdown **高さ維持ペナルティ**（0.79m から外れるとペナルティ）
height_weight = -2.0 #@param {type:"slider", min:-5.0, max:0.0, step:0.1}

#@markdown ---
#@markdown ### グループ C: 歩行品質
#@markdown **交互歩行ボーナス**（大きいほど左右交互のリズムを重視）
gait_weight = 0.5 #@param {type:"slider", min:0.0, max:2.0, step:0.1}

#@markdown **行動変化ペナルティ**（急な動きを抑える）
action_rate_weight = -0.05 #@param {type:"slider", min:-0.2, max:0.0, step:0.01}

#@markdown **トルクペナルティ**（省エネ）
torques_weight = -1e-4 #@param {type:"slider", min:-0.001, max:0.0, step:0.0001}

g1_reward_cfg = G1RewardConfig(
    lin_vel_weight=lin_vel_weight,
    ang_vel_weight=ang_vel_weight,
    orientation_weight=orientation_weight,
    height_weight=height_weight,
    gait_weight=gait_weight,
    action_rate_weight=action_rate_weight,
    torques_weight=torques_weight,
)

print('✅ G1 報酬設定完了')
print(f'  線速度追跡:  {lin_vel_weight}')
print(f'  角速度追跡:  {ang_vel_weight}')
print(f'  姿勢ペナルティ: {orientation_weight}')
print(f'  高さペナルティ: {height_weight}')
print(f'  歩行リズム:  {gait_weight}')
print(f'  行動変化:    {action_rate_weight}')

In [ ]:
#@title ⚙️ 学習設定

#@markdown ### 学習規模
#@markdown **学習ステップ数**（多いほど上手になるが時間がかかる）
total_timesteps = 500000 #@param {type:"slider", min:100000, max:2000000, step:100000}

#@markdown **並列環境数**（多いほど学習が速い — メモリと相談）
num_envs = 4 #@param {type:"slider", min:1, max:8, step:1}

#@markdown ---
#@markdown ### PPO ハイパーパラメータ
#@markdown **学習率**（通常は変えなくてOK）
learning_rate = 0.0003 #@param {type:"slider", min:0.0001, max:0.001, step:0.0001}

#@markdown **1回の更新で集めるステップ数**
n_steps = 2048 #@param {type:"slider", min:512, max:4096, step:256}

#@markdown **ミニバッチサイズ**
batch_size = 512 #@param {type:"slider", min:128, max:1024, step:128}

#@markdown ---
#@markdown ### ネットワーク構造
#@markdown **隠れ層のサイズ**（G1 は Go2 より大きいネットが必要）
net_arch_choice = '256-256-128 (推奨)' #@param ["512-256-128 (大)", "256-256-128 (推奨)", "128-128 (小)"]

arch_map = {
    '512-256-128 (大)': [512, 256, 128],
    '256-256-128 (推奨)': [256, 256, 128],
    '128-128 (小)': [128, 128],
}
net_arch = arch_map[net_arch_choice]

est_minutes_low  = total_timesteps // 100000 * 3
est_minutes_high = total_timesteps // 100000 * 10
print(f'学習設定:')
print(f'  ステップ数: {total_timesteps:,}')
print(f'  並列環境: {num_envs}')
print(f'  ネット構造: {net_arch}')
print(f'  ⏱ 目安: 約 {est_minutes_low}〜{est_minutes_high} 分 (GPU 有無で大きく変わります)')

In [ ]:
#@title 🚀 G1 歩行学習スタート！

import os, sys
os.chdir('/content/RoboQuest2026')
if '/content/RoboQuest2026' not in sys.path:
    sys.path.insert(0, '/content/RoboQuest2026')

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback

def _make_g1_env():
    env = G1WalkEnv(
        reward_config=g1_reward_cfg,
        max_episode_steps=500,
    )
    return Monitor(env, '/tmp/g1_monitor')

print('環境を作成中...')
vec_env = make_vec_env(_make_g1_env, n_envs=num_envs)
vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True)

model = PPO(
    'MlpPolicy',
    vec_env,
    learning_rate=learning_rate,
    n_steps=n_steps,
    batch_size=batch_size,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    policy_kwargs={'net_arch': net_arch},
    verbose=0,
)

total_params = sum(p.numel() for p in model.policy.parameters())
print(f'モデルパラメータ数: {total_params:,}')
print(f'\n🦿 G1 歩行学習を開始します... ({total_timesteps:,} ステップ)')

checkpoint_cb = CheckpointCallback(
    save_freq=max(50000 // num_envs, 1),
    save_path='/tmp/g1_checkpoints/',
    name_prefix='g1_walk',
)

model.learn(
    total_timesteps=total_timesteps,
    callback=checkpoint_cb,
    progress_bar=True,
)

G1_MODEL_PATH = '/tmp/g1_walk_model'
G1_VECNORM_PATH = '/tmp/g1_walk_vecnorm.pkl'
model.save(G1_MODEL_PATH)
vec_env.save(G1_VECNORM_PATH)

try:
    model.save(f'{SAVE_DIR}/g1_walk_model')
    vec_env.save(f'{SAVE_DIR}/g1_walk_vecnorm.pkl')
    print(f'\n💾 Google Drive に保存: {SAVE_DIR}')
except Exception:
    pass

print('\n✅ 学習完了！次のセルで学習曲線と動画を確認しましょう。')

In [ ]:
#@title 📈 学習曲線を確認しよう

import glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = ['IPAexGothic', 'Noto Sans CJK JP', 'DejaVu Sans']

monitor_files = glob.glob('/tmp/g1_monitor*.csv')
if not monitor_files:
    print('モニターファイルが見つかりません。学習を先に実行してください。')
else:
    dfs = []
    for f in monitor_files:
        try:
            df = pd.read_csv(f, skiprows=1)
            dfs.append(df)
        except Exception:
            pass

    if dfs:
        data = pd.concat(dfs).sort_values('t').reset_index(drop=True)
        data['episode'] = range(len(data))

        # 移動平均
        window = max(1, len(data) // 50)
        data['reward_smooth'] = data['r'].rolling(window=window, min_periods=1).mean()
        data['length_smooth'] = data['l'].rolling(window=window, min_periods=1).mean()

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        ax1.plot(data['episode'], data['r'], alpha=0.2, color='steelblue', linewidth=0.5)
        ax1.plot(data['episode'], data['reward_smooth'], color='steelblue', linewidth=2)
        ax1.set_xlabel('エピソード')
        ax1.set_ylabel('エピソード報酬')
        ax1.set_title('G1 歩行 学習曲線（報酬）')
        ax1.grid(True, alpha=0.3)

        ax2.plot(data['episode'], data['l'], alpha=0.2, color='darkorange', linewidth=0.5)
        ax2.plot(data['episode'], data['length_smooth'], color='darkorange', linewidth=2)
        ax2.set_xlabel('エピソード')
        ax2.set_ylabel('エピソード長さ [ステップ]')
        ax2.set_title('G1 歩行 学習曲線（生存ステップ数）')
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig('/tmp/g1_learning_curve.png', dpi=120, bbox_inches='tight')
        plt.show()

        n_ep = len(data)
        last_50 = data.tail(max(1, n_ep // 5))
        print(f'\n学習サマリー:')
        print(f'  総エピソード数: {n_ep}')
        print(f'  最終期平均報酬: {last_50["r"].mean():.2f} ± {last_50["r"].std():.2f}')
        print(f'  最終期平均生存: {last_50["l"].mean():.1f} ステップ')
    else:
        print('データが読み込めませんでした')

In [ ]:
#@title 🎬 G1 の歩行を動画で確認しよう

import mujoco
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecNormalize, DummyVecEnv

print('学習済みモデルを読み込み中...')
loaded_model = PPO.load(G1_MODEL_PATH)

# VecNormalize の統計を再現するためにダミー env を作成
_dummy_env = DummyVecEnv([lambda: G1WalkEnv(reward_config=g1_reward_cfg)])
eval_env = VecNormalize.load(G1_VECNORM_PATH, _dummy_env)
eval_env.training = False
eval_env.norm_reward = False

# 録画
raw_env = G1WalkEnv(
    reward_config=g1_reward_cfg,
    max_episode_steps=600,
    render_mode='rgb_array',
)
renderer = mujoco.Renderer(raw_env.model, height=480, width=640)
renderer.scene.flags[mujoco.mjtRndFlag.mjRND_SHADOW] = False

frames = []
heights = []
rewards_log = []

# 前向き歩行のコマンドで評価
obs_raw, _ = raw_env.reset()
raw_env.set_vel_cmd(vx=0.5, vy=0.0, omega=0.0)
raw_env.randomize_cmd = False

# eval_env の正規化統計を使って obs を変換
obs_vec = eval_env.normalize_obs(obs_raw[np.newaxis, :])

for step in range(600):
    action, _ = loaded_model.predict(obs_vec, deterministic=True)
    obs_raw, r, terminated, truncated, _ = raw_env.step(action[0])
    obs_vec = eval_env.normalize_obs(obs_raw[np.newaxis, :])

    heights.append(float(raw_env.data.qpos[2]))
    rewards_log.append(r)

    renderer.update_scene(raw_env.data)
    frames.append(renderer.render().copy())

    if terminated or truncated:
        print(f'  エピソード終了: {step + 1} ステップ')
        break

renderer.close()
raw_env.close()
eval_env.close()

print(f'\n録画フレーム数: {len(frames)}')
print(f'平均重心高さ: {np.mean(heights):.3f} m  (立位: {G1_STANDING_H} m)')
print(f'平均報酬/ステップ: {np.mean(rewards_log):.4f}')

# 動画表示
try:
    import mediapy as media
    media.show_video(frames, fps=30)
except ImportError:
    from matplotlib import animation
    from IPython.display import HTML, display
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.axis('off')
    img = ax.imshow(frames[0])
    ani = animation.FuncAnimation(
        fig, lambda f: [img.set_array(f)], frames=frames, interval=33, blit=True)
    plt.close(fig)
    display(HTML(ani.to_jshtml()))

# 重心高さの推移をプロット
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(heights, color='steelblue')
ax.axhline(G1_STANDING_H, color='green', linestyle='--', label=f'立位高さ ({G1_STANDING_H} m)')
ax.axhline(G1_FALL_H, color='red', linestyle='--', label=f'転倒閾値 ({G1_FALL_H} m)')
ax.set_xlabel('ステップ')
ax.set_ylabel('重心高さ [m]')
ax.set_title('G1 重心高さの推移')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#@title 🔬 関節ごとの動きを分析しよう

# 各関節がどのくらい動いているか可視化
import mujoco
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecNormalize, DummyVecEnv

loaded_model = PPO.load(G1_MODEL_PATH)
_dummy_env = DummyVecEnv([lambda: G1WalkEnv(reward_config=g1_reward_cfg)])
eval_env = VecNormalize.load(G1_VECNORM_PATH, _dummy_env)
eval_env.training = False
eval_env.norm_reward = False

raw_env = G1WalkEnv(reward_config=g1_reward_cfg, max_episode_steps=300)
obs_raw, _ = raw_env.reset()
raw_env.set_vel_cmd(vx=0.5, vy=0.0, omega=0.0)
raw_env.randomize_cmd = False

obs_vec = eval_env.normalize_obs(obs_raw[np.newaxis, :])
joint_traj = []

for _ in range(300):
    action, _ = loaded_model.predict(obs_vec, deterministic=True)
    obs_raw, _, terminated, truncated, _ = raw_env.step(action[0])
    obs_vec = eval_env.normalize_obs(obs_raw[np.newaxis, :])
    joint_traj.append(raw_env.data.qpos[7:7+G1_N_JOINTS].copy())
    if terminated or truncated:
        break

raw_env.close()
eval_env.close()

joint_traj = np.array(joint_traj)  # (T, 29)
T = len(joint_traj)
t_axis = np.arange(T) * G1_CTRL_DT

# 下半身（脚）の主要関節を表示
key_joints = {
    '左 hip_pitch': 0, '右 hip_pitch': 6,
    '左 knee': 3, '右 knee': 9,
    '左 ankle_pitch': 4, '右 ankle_pitch': 10,
}

fig, axes = plt.subplots(3, 2, figsize=(14, 9))
fig.suptitle('G1 脚の関節角度軌跡', fontsize=13)

for ax, (name, idx) in zip(axes.flatten(), key_joints.items()):
    ax.plot(t_axis, joint_traj[:, idx], linewidth=1.5)
    ax.axhline(G1_STANDING_CTRL[idx], color='gray', linestyle='--', alpha=0.6, label='立位')
    ax.set_xlabel('時間 [s]')
    ax.set_ylabel('角度 [rad]')
    ax.set_title(name)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/g1_joint_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ 関節分析完了')

In [ ]:
#@title 💾 モデルを保存して提出

import os, json

params = {
    'team_name': team_name,
    'robot': 'G1_29DOF',
    'task': 'velocity_tracking',
    'framework': 'stable-baselines3 PPO',
    'reward': {
        'lin_vel_weight': lin_vel_weight,
        'ang_vel_weight': ang_vel_weight,
        'orientation_weight': orientation_weight,
        'height_weight': height_weight,
        'gait_weight': gait_weight,
        'action_rate_weight': action_rate_weight,
        'torques_weight': torques_weight,
    },
    'training': {
        'total_timesteps': total_timesteps,
        'num_envs': num_envs,
        'learning_rate': learning_rate,
        'n_steps': n_steps,
        'batch_size': batch_size,
        'net_arch': net_arch,
    },
    'reference': 'https://github.com/unitreerobotics/unitree_rl_mjlab',
}

try:
    os.makedirs(SAVE_DIR, exist_ok=True)
    with open(f'{SAVE_DIR}/params.json', 'w', encoding='utf-8') as f:
        json.dump(params, f, ensure_ascii=False, indent=2)

    import shutil
    for fname in ['g1_learning_curve.png', 'g1_joint_analysis.png']:
        src = f'/tmp/{fname}'
        if os.path.exists(src):
            shutil.copy(src, f'{SAVE_DIR}/{fname}')

    print(f'✅ 保存完了！')
    print(f'  モデル:    {SAVE_DIR}/g1_walk_model.zip')
    print(f'  正規化統計: {SAVE_DIR}/g1_walk_vecnorm.pkl')
    print(f'  パラメータ: {SAVE_DIR}/params.json')
    print(f'  グラフ:    {SAVE_DIR}/g1_*.png')
    print(f'\n🎉 G1 歩行学習ノートブック 完了！')

except Exception as e:
    print(f'Drive 保存エラー: {e}')
    print('ローカル保存済み: /tmp/g1_walk_model.zip')